# DATASCAPE 2026 — ConvNeXt-Base grouped OOF teacher, A100 40 GB

This notebook trains one fixed-seed ConvNeXt teacher on rebuilt SigLIP2 duplicate groups.

- Only `development` rows are loaded for training and OOF prediction.
- The unseen, label-noisy holdout is never loaded into a dataset or evaluated.
- Every fold uses the same budget-selected fixed epoch count (6-9); validation labels never choose a checkpoint.
- Destructive blur, JPEG, downsample, and erasing augmentations are disabled.
- Outputs are original-label OOF probabilities plus a conservative cross-model pseudo-label audit. The audit does not modify a target.
- The runtime profile targets an A100 40 GB with 80 GB system RAM: BF16, channels-last, physical batch probing up to 128, effective batch 256, and up to 12 persistent loader workers.


## Before **Run all** on the paid runtime

1. Select an **A100 40 GB** runtime. The notebook stops before training if the GPU or system RAM is below the required profile.
2. Make sure the Kaggle account behind the embedded token has accepted the competition rules. Authentication and dataset access run before model training.
3. Approve the Google Drive mount promptly. Completed-fold OOF files and contracts are saved under `MyDrive/DATASCAPE/convnext_grouped_oof_teacher_seed42`.
4. Keep internet access enabled for the Kaggle dataset and public timm weights. Fresh grouping and SigLIP OOF files are loaded from `MyDrive/DATASCAPE/siglip2_first_pipeline` and hash-checked before training.

The batch probe tries 128 first and falls back through exact divisors of effective batch 256. A short training benchmark then chooses one fixed 6-9 epoch count for every fold within the 88-minute working budget, including time already spent on setup, with 10 minutes reserved for final OOF inference and writes.


In [ ]:
# Kaggle API bootstrap for one-click Colab fallback.
import os
import time
from pathlib import Path

RUN_STARTED_AT = time.monotonic()

EMBEDDED_GEOMETRY_CANDIDATES = 5000
REQUIRED_GEOMETRY_CANDIDATES = 25000
EMBEDDED_GROUPING_PROTOCOL_VERSION = 'legacy_pre_component_holdout_audit'
REQUIRED_GROUPING_PROTOCOL_VERSION = 'siglip2_geometry_25000_component_split_v2'
KAGGLE_API_TOKEN = 'KGAT_c2a604d54464b1bc7271fb95428085f2'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

kaggle_directory = Path.home() / '.kaggle'
kaggle_directory.mkdir(parents=True, exist_ok=True)
(kaggle_directory / 'access_token').write_text(KAGGLE_API_TOKEN + '\n')
(kaggle_directory / 'access_token').chmod(0o600)
print('Kaggle API configured for this runtime.')


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    'numpy': 'numpy', 'pandas': 'pandas', 'PIL': 'Pillow',
    'pillow_avif': 'pillow-avif-plugin', 'sklearn': 'scikit-learn',
    'torch': 'torch', 'torchvision': 'torchvision', 'timm': 'timm',
    'tqdm': 'tqdm',
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import base64
import contextlib
import gc
import gzip
import hashlib
import io
import json
import math
import random
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pillow_avif
import timm
import torch
from PIL import Image, ImageOps
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support
from timm.utils import ModelEmaV3
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules
print('Colab:', IN_COLAB, '| Python:', sys.version.split()[0], '| torch:', torch.__version__)


## 1. Fixed experiment configuration

The model, training seed, and effective batch match the original-label ConvNeXt-Base baseline. Fold selection is fixed by the embedded grouped manifest.


In [ ]:
SEED = 42
CLASSES = ('Aman', 'Rusak', 'Lainnya')
CLASS_TO_INDEX = {label: index for index, label in enumerate(CLASSES)}
COMPETITION = 'datascape-2026-data-mining-competition'

MODEL_ID = 'hf_hub:timm/convnext_base.fb_in22k_ft_in1k_384'
MODEL_REVISION_NOTE = 'timm Hugging Face model ID; cache the resolved weights with the run artifacts'
IMAGE_SIZE = 384
FOLDS = 5
MIN_FIXED_EPOCHS = 6
MAX_FIXED_EPOCHS = 9
RUN_BUDGET_MINUTES = 88
FINAL_OUTPUT_RESERVE_MINUTES = 10
PERFORMANCE_WARMUP_BATCHES = 4
PERFORMANCE_MEASURE_BATCHES = 24
PERFORMANCE_SAFETY_FACTOR = 1.15
TARGET_EFFECTIVE_BATCH_SIZE = 256
BATCH_CANDIDATES = (128, 64, 32, 16, 8, 4, 2)
MPS_BATCH_SIZE = 2
MAX_MEMORY_FRACTION = 0.92
REQUIRE_A100_40GB = True
MIN_SYSTEM_RAM_GIB = 60
REFERENCE_BATCH_SIZE = 128
REFERENCE_LR = 2e-4
MAX_LR = 2.5e-4
MIN_LR = 1e-6
WARMUP_EPOCHS = 1
WEIGHT_DECAY = 5e-2
LABEL_SMOOTHING = 0.08
BALANCE_STRENGTH = 0.50
GRAD_CLIP_NORM = 1.0
DROP_RATE = 0.20
DROP_PATH_RATE = 0.20
MODEL_EMA_DECAY = 0.9997
USE_HORIZONTAL_FLIP_TTA = True
SAVE_FINAL_CHECKPOINTS = False
RESUME_COMPLETED_FOLDS = True

# Only mild, non-destructive transforms.
HORIZONTAL_FLIP_PROBABILITY = 0.50
AFFINE_DEGREES = 3.0
AFFINE_TRANSLATE = (0.02, 0.02)
AFFINE_SCALE = (0.97, 1.03)
COLOR_JITTER = 0.06

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

PROJECT_ROOT = Path.cwd()
LOCAL_DATA_ROOT = PROJECT_ROOT / COMPETITION
DOWNLOAD_ROOT = Path('/content/datascape_dataset')
LOCAL_OUTPUT_DIR = PROJECT_ROOT / 'artifacts/convnext_grouped_oof_teacher_seed42'
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/DATASCAPE/convnext_grouped_oof_teacher_seed42')
USE_GOOGLE_DRIVE = True

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
else:
    OUTPUT_DIR = LOCAL_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print('Output:', OUTPUT_DIR)


## 2. Resolve images and verify the rebuilt grouped manifest

The grouped manifest and SigLIP2 development OOF are loaded from the completed Drive pipeline. Its protocol version, geometry-candidate count, hashes, and structural invariants are asserted before any image is loaded.


In [ ]:
GROUPED_MANIFEST_GZIP_BASE64 = '''...'''
EMBEDDED_GROUPED_MANIFEST_SHA256 = 'd32c1571b676344d8f5598e1f6c0849f59fbf896b96ae7eed2d0a8760f4c3f4c'
SIGLIP_AUDIT_GZIP_BASE64 = '''...'''
EMBEDDED_SIGLIP_AUDIT_SHA256 = 'c4c53e06192e1296098513ca6f0dcc0f4bd7a69e6b1fc0a46727f594c76238b0'


def decode_embedded_csv(encoded, expected_sha256):
    raw = gzip.decompress(base64.b64decode(encoded))
    actual = hashlib.sha256(raw).hexdigest()
    if actual != expected_sha256:
        raise RuntimeError(f'Embedded CSV hash mismatch: {actual} != {expected_sha256}')
    return pd.read_csv(io.BytesIO(raw))


def safe_extract(zip_path, destination):
    zip_path, destination = Path(zip_path), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)
    return destination


def find_dataset_root(start):
    start = Path(start).expanduser().resolve()
    candidates = [start]
    if start.is_dir():
        candidates.extend(path.parent for path in start.rglob('sample_submission.csv'))
    for root in candidates:
        if (root / 'sample_submission.csv').is_file() and (root / 'Train_Set').is_dir():
            if all((root / 'Train_Set' / label).is_dir() for label in CLASSES):
                return root
    raise FileNotFoundError(f'Dataset root not found under {start}')


def resolve_dataset_root():
    for candidate in (LOCAL_DATA_ROOT, Path('/content') / COMPETITION, DOWNLOAD_ROOT):
        if candidate.exists():
            try:
                return find_dataset_root(candidate)
            except FileNotFoundError:
                pass
    if importlib.util.find_spec('kagglehub') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub
    from kagglehub.config import set_kaggle_api_token
    set_kaggle_api_token(KAGGLE_API_TOKEN)
    identity = kagglehub.whoami(verbose=False)
    username = identity.get('username', identity) if isinstance(identity, dict) else identity
    print('Kaggle authenticated as:', username)
    print('Dataset is not local; validating competition access by downloading it now.')
    downloaded = Path(kagglehub.competition_download(COMPETITION))
    if downloaded.is_file() and downloaded.suffix.lower() == '.zip':
        return find_dataset_root(safe_extract(downloaded, DOWNLOAD_ROOT))
    return find_dataset_root(downloaded)


fresh_pipeline_dir = (
    Path('/content/drive/MyDrive/DATASCAPE/siglip2_first_pipeline')
    if IN_COLAB else PROJECT_ROOT / 'artifacts/siglip2_first_pipeline'
)
fresh_manifest_path = fresh_pipeline_dir / 'siglip2_grouped_manifest.csv'
fresh_oof_path = fresh_pipeline_dir / 'siglip2_linear_probe_oof.csv'
fresh_stability_path = fresh_pipeline_dir / 'siglip2_probe_partition_stability.csv'
fresh_summary_path = fresh_pipeline_dir / 'stage5_summary.json'
fresh_paths = (fresh_manifest_path, fresh_oof_path, fresh_stability_path, fresh_summary_path)

if all(path.is_file() for path in fresh_paths):
    fresh_summary = json.loads(fresh_summary_path.read_text())
    protocol = fresh_summary.get('grouping_protocol_version')
    candidates = int(fresh_summary.get('geometry_candidates_retained', 0))
    if protocol != REQUIRED_GROUPING_PROTOCOL_VERSION or candidates < REQUIRED_GEOMETRY_CANDIDATES:
        raise RuntimeError(
            f'Fresh Drive artifacts are not ready: protocol={protocol}, '
            f'geometry_candidates={candidates}. Complete the revised SigLIP2 notebook first.'
        )
    grouped_manifest = pd.read_csv(fresh_manifest_path)[
        ['filepath', 'label', 'dedup_group', 'partition', 'fold']
    ]
    siglip_audit = pd.read_csv(fresh_oof_path).merge(
        pd.read_csv(fresh_stability_path)[
            ['filepath', 'partition_unanimous', 'partition_consensus_rate']
        ],
        on='filepath', validate='one_to_one',
    )
    manifest_raw = grouped_manifest.to_csv(index=False).encode()
    siglip_raw = siglip_audit.to_csv(index=False).encode()
    GROUPED_MANIFEST_SHA256 = hashlib.sha256(manifest_raw).hexdigest()
    SIGLIP_AUDIT_SHA256 = hashlib.sha256(siglip_raw).hexdigest()
    print('Loaded fresh grouping protocol from Drive:', protocol, '| candidates:', candidates)
else:
    raise RuntimeError(
        'Fresh SigLIP2 grouping artifacts are missing from ' + str(fresh_pipeline_dir) + '. '
        'Complete the revised SigLIP2 notebook before starting paid ConvNeXt training.'
    )

DATA_ROOT = resolve_dataset_root()

required = {'filepath', 'label', 'dedup_group', 'partition', 'fold'}
missing = required - set(grouped_manifest.columns)
if missing:
    raise ValueError(f'Grouped manifest missing columns: {sorted(missing)}')
if len(grouped_manifest) != 12224:
    raise ValueError(f'Expected 12224 grouped rows, got {len(grouped_manifest)}')
if grouped_manifest.groupby('dedup_group').partition.nunique().max() != 1:
    raise ValueError('A duplicate group crosses development/holdout.')
development_mask = grouped_manifest.partition.eq('development')
if grouped_manifest.loc[development_mask].groupby('dedup_group').fold.nunique().max() != 1:
    raise ValueError('A duplicate group crosses OOF folds.')
if not grouped_manifest.loc[~development_mask, 'fold'].eq(-1).all():
    raise ValueError('Unseen holdout must have fold -1.')
if set(grouped_manifest.loc[development_mask, 'fold']) != set(range(FOLDS)):
    raise ValueError('Development fold assignment is incomplete.')

holdout_rows = int((~development_mask).sum())
holdout_group_hash = hashlib.sha256(
    '\n'.join(sorted(grouped_manifest.loc[~development_mask, 'dedup_group'].unique())).encode()
).hexdigest()
train_frame = grouped_manifest.loc[development_mask].copy().reset_index(drop=True)
train_frame['fold'] = train_frame.fold.astype(int)
train_frame['label_original'] = train_frame.label
train_frame['image_path'] = train_frame.filepath.map(lambda value: str(DATA_ROOT / value))
missing_paths = train_frame.loc[
    ~train_frame.image_path.map(lambda value: Path(value).is_file()), 'filepath'
]
if len(missing_paths):
    raise FileNotFoundError(f'{len(missing_paths)} development images missing; first: {missing_paths.iloc[0]}')
if set(siglip_audit.filepath) != set(train_frame.filepath):
    raise ValueError('Embedded SigLIP OOF rows do not align with development rows.')

# Holdout labels and paths are deliberately removed from the live training state.
del grouped_manifest, development_mask
print('Dataset:', DATA_ROOT)
print('Development rows:', len(train_frame), '| unseen label-noisy holdout rows:', holdout_rows)
print(pd.crosstab(train_frame.fold, train_frame.label).reindex(columns=CLASSES, fill_value=0))


## 3. Image pipeline

Letterboxing preserves aspect ratio. Training uses only flip, small affine motion, and mild color jitter; no blur, JPEG degradation, downsampling, or erasing is applied.


In [ ]:
class Letterbox:
    def __init__(self, size):
        self.size = int(size)
        self.fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)

    def __call__(self, image):
        return ImageOps.pad(
            image, (self.size, self.size), Image.Resampling.BILINEAR,
            color=self.fill, centering=(0.5, 0.5),
        )


def load_rgb_image(path):
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened)
        if image.mode == 'P':
            image = image.convert('RGBA')
        if image.mode in {'RGBA', 'LA'} or 'transparency' in image.info:
            rgba = image.convert('RGBA')
            fill = tuple(int(value * 255) for value in IMAGENET_MEAN) + (255,)
            image = Image.alpha_composite(Image.new('RGBA', rgba.size, fill), rgba).convert('RGB')
        else:
            image = image.convert('RGB')
        return image.copy()


fill = tuple(int(round(value * 255)) for value in IMAGENET_MEAN)
TRAIN_TRANSFORM = transforms.Compose([
    Letterbox(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=HORIZONTAL_FLIP_PROBABILITY),
    transforms.RandomAffine(
        degrees=AFFINE_DEGREES, translate=AFFINE_TRANSLATE,
        scale=AFFINE_SCALE, interpolation=transforms.InterpolationMode.BILINEAR,
        fill=fill,
    ),
    transforms.ColorJitter(
        brightness=COLOR_JITTER, contrast=COLOR_JITTER,
        saturation=COLOR_JITTER, hue=0.01,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
EVAL_TRANSFORM = transforms.Compose([
    Letterbox(IMAGE_SIZE), transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class PackageDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.transform = TRAIN_TRANSFORM if training else EVAL_TRANSFORM

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = self.transform(load_rgb_image(row.image_path))
        return image, CLASS_TO_INDEX[row.label_original], index


NUM_WORKERS = min(12, os.cpu_count() or 4) if IN_COLAB else 0
PIN_MEMORY = torch.cuda.is_available()


def balanced_sampler(frame, seed):
    counts = frame.label_original.value_counts()
    weights = frame.label_original.map(
        lambda label: counts[label] ** -BALANCE_STRENGTH
    ).to_numpy(copy=True)
    generator = torch.Generator().manual_seed(seed)
    return WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.double), len(frame),
        replacement=True, generator=generator,
    )


def make_loader(dataset, batch_size, sampler=None):
    options = dict(
        dataset=dataset, batch_size=batch_size, sampler=sampler,
        shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(**options)


preview = PackageDataset(train_frame.head(2), training=True)
print('Preview:', tuple(preview[0][0].shape), preview[0][0].dtype)


## 4. Budgeted fixed-epoch ConvNeXt training

The notebook benchmarks the actual A100 training path, then selects one fixed count from 6 through 9 epochs that fits the paid runtime budget. Every fold uses that same count. Validation is run exactly once at the end of each fold, so validation labels cannot select a checkpoint and GPU time is not spent on redundant per-epoch inference.


In [ ]:
DEVICE = torch.device(
    'cuda' if torch.cuda.is_available()
    else ('mps' if torch.backends.mps.is_available() else 'cpu')
)
IS_CUDA = DEVICE.type == 'cuda'
IS_MPS = DEVICE.type == 'mps'

if IS_CUDA:
    GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    GPU_NAME = torch.cuda.get_device_name(0)
    COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
    USE_BF16 = COMPUTE_CAPABILITY[0] >= 8 and torch.cuda.is_bf16_supported()
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    USE_GRAD_SCALER = AMP_DTYPE == torch.float16
    USE_GRAD_CHECKPOINTING = GPU_VRAM_GIB <= 28
else:
    GPU_VRAM_GIB = 0.0
    GPU_NAME = str(DEVICE)
    AMP_DTYPE = torch.float32
    USE_GRAD_SCALER = False
    USE_GRAD_CHECKPOINTING = IS_MPS

try:
    SYSTEM_RAM_GIB = (
        os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024 ** 3)
    )
except (AttributeError, ValueError):
    SYSTEM_RAM_GIB = float('nan')

if REQUIRE_A100_40GB:
    if not IS_CUDA or 'A100' not in GPU_NAME.upper() or GPU_VRAM_GIB < 38:
        raise RuntimeError(
            f'Paid-run guard: expected A100 with about 40 GB VRAM, got '
            f'{GPU_NAME} with {GPU_VRAM_GIB:.1f} GiB.'
        )
    if np.isfinite(SYSTEM_RAM_GIB) and SYSTEM_RAM_GIB < MIN_SYSTEM_RAM_GIB:
        raise RuntimeError(
            f'Paid-run guard: expected at least {MIN_SYSTEM_RAM_GIB} GiB system RAM, '
            f'got {SYSTEM_RAM_GIB:.1f} GiB.'
        )
print(
    'Resource preflight:', GPU_NAME, f'| VRAM={GPU_VRAM_GIB:.1f} GiB',
    f'| system RAM={SYSTEM_RAM_GIB:.1f} GiB', '| AMP=', AMP_DTYPE,
)


def autocast_context():
    if IS_CUDA:
        return torch.autocast('cuda', dtype=AMP_DTYPE)
    return contextlib.nullcontext()


def build_model():
    model = timm.create_model(
        MODEL_ID, pretrained=True, num_classes=len(CLASSES),
        drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE,
    )
    if USE_GRAD_CHECKPOINTING:
        model.set_grad_checkpointing(True)
    model = model.to(DEVICE)
    if IS_CUDA:
        model = model.to(memory_format=torch.channels_last)
    return model


def prepare_images(images):
    images = images.to(DEVICE, non_blocking=IS_CUDA)
    if IS_CUDA:
        images = images.contiguous(memory_format=torch.channels_last)
    return images


def probe_cuda_batch():
    if not IS_CUDA:
        return MPS_BATCH_SIZE if IS_MPS else 2
    maximum = 8 if GPU_VRAM_GIB <= 18 else (16 if GPU_VRAM_GIB <= 28 else 128)
    for batch in (value for value in BATCH_CANDIDATES if value <= maximum):
        model = ema = optimizer = scaler = images = targets = loss = None
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            model = build_model()
            ema = ModelEmaV3(
                model, decay=MODEL_EMA_DECAY, use_warmup=True,
                warmup_gamma=1.0, warmup_power=2 / 3,
            )
            optimizer = torch.optim.AdamW(model.parameters(), lr=REFERENCE_LR, weight_decay=WEIGHT_DECAY)
            scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
            images = torch.randn(batch, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
            targets = torch.randint(0, len(CLASSES), (batch,), device=DEVICE)
            with autocast_context():
                loss = nn.functional.cross_entropy(model(images), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ema.update(model, step=1)
            torch.cuda.synchronize()
            fraction = torch.cuda.max_memory_reserved() / torch.cuda.get_device_properties(0).total_memory
            print(f'batch {batch}: {fraction:.1%} VRAM')
            if fraction <= MAX_MEMORY_FRACTION:
                print('Selected physical batch:', batch)
                return batch
        except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
            if 'out of memory' not in str(error).lower():
                raise
            print('OOM at batch', batch)
        finally:
            del model, ema, optimizer, scaler, images, targets, loss
            torch.cuda.empty_cache()
    raise RuntimeError('ConvNeXt-Base does not fit at batch size 2.')


BATCH_SIZE = probe_cuda_batch()
GRAD_ACCUM_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BATCH_SIZE / BATCH_SIZE))
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS
EVAL_BATCH_SIZE = max(4, min(128, BATCH_SIZE * 2))
BASE_LR = min(MAX_LR, REFERENCE_LR * math.sqrt(EFFECTIVE_BATCH_SIZE / REFERENCE_BATCH_SIZE))


def existing_epoch_contract():
    paths = sorted(OUTPUT_DIR.glob('fold_*/complete.json'))
    if not paths:
        return None
    contracts = [json.loads(path.read_text()) for path in paths]
    epochs = {int(item['fixed_epochs']) for item in contracts}
    if len(epochs) != 1:
        raise RuntimeError(f'Completed folds disagree on fixed epochs: {sorted(epochs)}')
    for item in contracts:
        if item['manifest_sha256'] != GROUPED_MANIFEST_SHA256:
            raise RuntimeError('Completed fold manifest hash does not match this notebook.')
        if item['model_id'] != MODEL_ID or int(item['seed']) != SEED:
            raise RuntimeError('Completed fold model/seed contract does not match this notebook.')
    return epochs.pop()


def estimate_training_epoch_seconds():
    probe_frame = train_frame.loc[train_frame.fold.ne(0)].copy()
    loader = make_loader(
        PackageDataset(probe_frame, training=True), BATCH_SIZE,
        balanced_sampler(probe_frame, SEED + 9000),
    )
    torch.manual_seed(SEED + 9000)
    model = build_model()
    ema = ModelEmaV3(
        model, decay=MODEL_EMA_DECAY, use_warmup=True,
        warmup_gamma=1.0, warmup_power=2 / 3,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
    optimizer.zero_grad(set_to_none=True)
    measured_batches = 0
    started = None
    optimizer_updates = 0
    limit = PERFORMANCE_WARMUP_BATCHES + PERFORMANCE_MEASURE_BATCHES
    for step, (images, targets, _) in enumerate(tqdm(loader, desc='A100 throughput preflight')):
        if step == PERFORMANCE_WARMUP_BATCHES:
            if IS_CUDA:
                torch.cuda.synchronize()
            started = time.monotonic()
        images = prepare_images(images)
        targets = targets.to(DEVICE, non_blocking=IS_CUDA)
        with autocast_context():
            loss = criterion(model(images), targets) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            optimizer_updates += 1
            ema.update(model, step=optimizer_updates)
        if step >= PERFORMANCE_WARMUP_BATCHES:
            measured_batches += 1
        if step + 1 >= limit:
            break
    if IS_CUDA:
        torch.cuda.synchronize()
    measured_seconds = time.monotonic() - started
    estimate = (
        measured_seconds / measured_batches * len(loader) * PERFORMANCE_SAFETY_FACTOR
    )
    del model, ema, optimizer, scaler, loader
    gc.collect()
    if IS_CUDA:
        torch.cuda.empty_cache()
    return float(estimate)


contract_epochs = existing_epoch_contract()
if contract_epochs is not None:
    FIXED_EPOCHS = contract_epochs
    ESTIMATED_EPOCH_SECONDS = None
    print('Resume contract fixes every fold at', FIXED_EPOCHS, 'epochs.')
else:
    ESTIMATED_EPOCH_SECONDS = estimate_training_epoch_seconds()
    elapsed_seconds = time.monotonic() - RUN_STARTED_AT
    available_training_seconds = (
        RUN_BUDGET_MINUTES * 60
        - elapsed_seconds
        - FINAL_OUTPUT_RESERVE_MINUTES * 60
    )
    affordable_epochs = math.floor(
        available_training_seconds / (FOLDS * ESTIMATED_EPOCH_SECONDS)
    )
    if affordable_epochs < MIN_FIXED_EPOCHS:
        raise RuntimeError(
            f'Paid-run guard: measured {ESTIMATED_EPOCH_SECONDS / 60:.2f} min/epoch, '
            f'so fewer than {MIN_FIXED_EPOCHS} epochs/fold fit in the remaining budget. '
            'Stop before training and start a fresh A100 runtime.'
        )
    FIXED_EPOCHS = min(MAX_FIXED_EPOCHS, affordable_epochs)
    print(
        f'Budget preflight: estimated {ESTIMATED_EPOCH_SECONDS / 60:.2f} min/epoch; '
        f'selected {FIXED_EPOCHS} fixed epochs/fold within {RUN_BUDGET_MINUTES} minutes.'
    )

print(
    'Device:', DEVICE, '| batch:', BATCH_SIZE, '| accumulation:', GRAD_ACCUM_STEPS,
    '| effective batch:', EFFECTIVE_BATCH_SIZE, '| eval batch:', EVAL_BATCH_SIZE,
    '| fixed epochs:', FIXED_EPOCHS, '| loader workers:', NUM_WORKERS,
)


def learning_rate_at(epoch):
    if epoch < WARMUP_EPOCHS:
        return BASE_LR * (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(FIXED_EPOCHS - WARMUP_EPOCHS - 1, 1)
    return MIN_LR + (BASE_LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(progress, 1)))


@torch.inference_mode()
def predict(model, loader, use_tta=True):
    model.eval()
    probabilities, targets_all, indices_all = [], [], []
    for images, targets, indices in tqdm(loader, desc='predict', leave=False):
        images = prepare_images(images)
        with autocast_context():
            probs = model(images).softmax(1)
            if use_tta:
                probs = 0.5 * (probs + model(torch.flip(images, dims=[3])).softmax(1))
        probabilities.append(probs.float().cpu().numpy())
        targets_all.append(targets.numpy())
        indices_all.append(indices.numpy())
    order = np.argsort(np.concatenate(indices_all))
    return {
        'probabilities': np.concatenate(probabilities)[order],
        'targets': np.concatenate(targets_all)[order],
    }


def train_fold(fold):
    train = train_frame.loc[train_frame.fold.ne(fold)].copy()
    valid = train_frame.loc[train_frame.fold.eq(fold)].copy()
    train_loader = make_loader(
        PackageDataset(train, training=True), BATCH_SIZE,
        balanced_sampler(train, SEED + fold),
    )
    valid_loader = make_loader(PackageDataset(valid, training=False), EVAL_BATCH_SIZE)

    torch.manual_seed(SEED + fold)
    model = build_model()
    ema = ModelEmaV3(
        model, decay=MODEL_EMA_DECAY, use_warmup=True,
        warmup_gamma=1.0, warmup_power=2 / 3,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_GRAD_SCALER)
    history = []
    optimizer_updates = 0

    for epoch in range(FIXED_EPOCHS):
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group['lr'] = lr
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss, total, started = 0.0, 0, time.time()
        number_of_batches = len(train_loader)
        progress = tqdm(train_loader, desc=f'fold {fold} epoch {epoch + 1}', leave=False)
        for step, (images, targets, _) in enumerate(progress):
            images = prepare_images(images)
            targets = targets.to(DEVICE, non_blocking=IS_CUDA)
            window_start = (step // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
            window_size = min(GRAD_ACCUM_STEPS, number_of_batches - window_start)
            with autocast_context():
                raw_loss = criterion(model(images), targets)
                loss = raw_loss / window_size
            scaler.scale(loss).backward()
            should_step = (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == number_of_batches
            if should_step:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                optimizer_updates += 1
                ema.update(model, step=optimizer_updates)
            total_loss += float(raw_loss.item()) * len(images)
            total += len(images)

        row = {
            'fold': fold, 'epoch': epoch + 1, 'lr': lr,
            'train_loss': total_loss / total,
            'seconds': time.time() - started,
        }
        history.append(row)
        print(
            f"fold {fold} epoch {epoch + 1}/{FIXED_EPOCHS}: "
            f"loss={row['train_loss']:.4f} "
            f"time={row['seconds'] / 60:.1f}m"
        )

    # Fixed final epoch; no validation-dependent checkpoint selection.
    final_model = ema.module
    final_result = predict(final_model, valid_loader, use_tta=USE_HORIZONTAL_FLIP_TTA)
    del model, ema, optimizer, scaler, train_loader, valid_loader
    gc.collect()
    if IS_CUDA:
        torch.cuda.empty_cache()
    elif IS_MPS:
        torch.mps.empty_cache()
    return valid, final_result['probabilities'], pd.DataFrame(history), final_model


## 5. Five-fold OOF generation

Each completed fold writes filepath-keyed probabilities. A restarted runtime reuses only fold files whose filepath set exactly matches the embedded assignment.


In [ ]:
oof_probabilities = np.full((len(train_frame), len(CLASSES)), np.nan, dtype=np.float32)
history_rows = []

for fold in range(FOLDS):
    fold_dir = OUTPUT_DIR / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    fold_oof_path = fold_dir / 'oof.csv'
    complete_path = fold_dir / 'complete.json'
    expected = train_frame.loc[train_frame.fold.eq(fold)].copy()

    if RESUME_COMPLETED_FOLDS and fold_oof_path.is_file() and complete_path.is_file():
        contract = json.loads(complete_path.read_text())
        expected_contract = {
            'fold': fold,
            'manifest_sha256': GROUPED_MANIFEST_SHA256,
            'model_id': MODEL_ID,
            'seed': SEED,
            'fixed_epochs': FIXED_EPOCHS,
            'effective_batch_size': EFFECTIVE_BATCH_SIZE,
        }
        mismatches = {
            key: (contract.get(key), value)
            for key, value in expected_contract.items()
            if contract.get(key) != value
        }
        if mismatches:
            raise RuntimeError(f'Fold {fold} resume contract mismatch: {mismatches}')
        saved = pd.read_csv(fold_oof_path)
        if set(saved.filepath) != set(expected.filepath):
            raise ValueError(f'Fold {fold} resume file does not match the embedded group manifest.')
        saved = expected[['filepath']].merge(saved, on='filepath', validate='one_to_one')
        probabilities = saved[[f'p_{label}' for label in CLASSES]].to_numpy(np.float32)
        if not np.isfinite(probabilities).all():
            raise RuntimeError(f'Fold {fold} resume probabilities are not finite.')
        print('Resumed fold', fold, 'from', fold_oof_path)
    else:
        valid, probabilities, history, final_model = train_fold(fold)
        if list(valid.filepath) != list(expected.filepath):
            raise RuntimeError(f'Fold {fold} validation order changed unexpectedly.')
        saved = valid[['filepath', 'label_original', 'dedup_group', 'fold']].copy()
        for class_index, label in enumerate(CLASSES):
            saved[f'p_{label}'] = probabilities[:, class_index]
        saved.to_csv(fold_oof_path, index=False)
        history.to_csv(fold_dir / 'history.csv', index=False)
        history_rows.append(history)
        if SAVE_FINAL_CHECKPOINTS:
            torch.save({
                'model_state': {key: value.cpu() for key, value in final_model.state_dict().items()},
                'fold': fold, 'fixed_epochs': FIXED_EPOCHS, 'seed': SEED,
                'manifest_sha256': GROUPED_MANIFEST_SHA256,
            }, fold_dir / 'final_epoch.pt')
        completion = {
            'fold': fold,
            'manifest_sha256': GROUPED_MANIFEST_SHA256,
            'model_id': MODEL_ID,
            'seed': SEED,
            'fixed_epochs': FIXED_EPOCHS,
            'physical_batch_size': BATCH_SIZE,
            'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
            'effective_batch_size': EFFECTIVE_BATCH_SIZE,
            'validation_rows': int(len(saved)),
        }
        temporary_complete_path = complete_path.with_suffix('.json.tmp')
        temporary_complete_path.write_text(json.dumps(completion, indent=2) + '\n')
        temporary_complete_path.replace(complete_path)
        del final_model
        gc.collect()

    global_indices = train_frame.index[train_frame.fold.eq(fold)].to_numpy()
    oof_probabilities[global_indices] = probabilities

if not np.isfinite(oof_probabilities).all():
    raise RuntimeError('OOF matrix contains missing or non-finite probabilities.')

np.save(OUTPUT_DIR / 'convnext_grouped_oof.npy', oof_probabilities)
oof_frame = train_frame[['filepath', 'label_original', 'dedup_group', 'fold']].copy()
for class_index, label in enumerate(CLASSES):
    oof_frame[f'p_{label}'] = oof_probabilities[:, class_index]
oof_frame.to_csv(OUTPUT_DIR / 'convnext_grouped_oof.csv', index=False)
if history_rows:
    pd.concat(history_rows, ignore_index=True).to_csv(OUTPUT_DIR / 'training_history_new_folds.csv', index=False)
print('Saved OOF:', OUTPUT_DIR / 'convnext_grouped_oof.csv')


## 6. Original-label evaluation and SigLIP agreement

All metrics use the untouched original labels. The pseudo-label audit is conservative and only proposes Aman↔Rusak candidates when both teachers agree, SigLIP is partition-unanimous, both reject Lainnya, and the duplicate component has no conflicting proposed target. It writes suggestions only.


In [ ]:
targets = train_frame.label_original.map(CLASS_TO_INDEX).to_numpy()
predicted = oof_probabilities.argmax(1)
precision, recall, f1, support = precision_recall_fscore_support(
    targets, predicted, labels=range(len(CLASSES)), zero_division=0,
)
class_metrics = pd.DataFrame({
    'label': CLASSES, 'precision': precision, 'recall': recall,
    'f1': f1, 'support': support,
})
class_metrics.to_csv(OUTPUT_DIR / 'convnext_grouped_oof_class_metrics.csv', index=False)

hard = confusion_matrix(targets, predicted, labels=range(len(CLASSES)))
pd.DataFrame(hard, index=CLASSES, columns=CLASSES).rename_axis('observed_label').to_csv(
    OUTPUT_DIR / 'convnext_grouped_oof_confusion_counts.csv'
)
pd.DataFrame(
    hard / hard.sum(1, keepdims=True), index=CLASSES, columns=CLASSES
).rename_axis('observed_label').to_csv(
    OUTPUT_DIR / 'convnext_grouped_oof_confusion_row_normalized.csv'
)
print(class_metrics.to_string(index=False))
print('Macro F1:', float(f1.mean()))

audit = oof_frame.merge(
    siglip_audit.drop(columns=['label', 'primary_fold']),
    on='filepath', suffixes=('_convnext', '_siglip'), validate='one_to_one',
)
convnext_columns = [f'p_{label}_convnext' for label in CLASSES]
siglip_columns = [f'p_{label}_siglip' for label in CLASSES]
convnext_probs = audit[convnext_columns].to_numpy()
siglip_probs = audit[siglip_columns].to_numpy()
class_array = np.asarray(CLASSES)
audit['convnext_prediction'] = class_array[convnext_probs.argmax(1)]
audit['siglip_prediction'] = class_array[siglip_probs.argmax(1)]
audit['convnext_confidence'] = convnext_probs.max(1)
audit['siglip_confidence'] = siglip_probs.max(1)
audit['cross_model_agreement'] = audit.convnext_prediction.eq(audit.siglip_prediction)

audit['candidate_target'] = np.where(
    audit.cross_model_agreement, audit.convnext_prediction, ''
)
base_candidate = (
    audit.label_original.isin(['Aman', 'Rusak'])
    & audit.candidate_target.isin(['Aman', 'Rusak'])
    & audit.candidate_target.ne(audit.label_original)
    & audit.partition_unanimous.astype(bool)
    & audit.convnext_confidence.ge(0.85)
    & audit.siglip_confidence.ge(0.80)
    & audit.p_Lainnya_convnext.lt(0.30)
    & audit.p_Lainnya_siglip.lt(0.30)
)
audit['base_candidate'] = base_candidate

candidate_targets_per_group = audit.loc[base_candidate].groupby(
    'dedup_group'
).candidate_target.nunique()
groups_with_lainnya = set(
    audit.loc[audit.label_original.eq('Lainnya'), 'dedup_group']
)
audit['component_target_consistent'] = audit.dedup_group.map(
    candidate_targets_per_group
).fillna(0).le(1)
audit['component_contains_lainnya'] = audit.dedup_group.isin(groups_with_lainnya)
audit['pseudo_label_candidate'] = (
    audit.base_candidate
    & audit.component_target_consistent
    & ~audit.component_contains_lainnya
)

audit.to_csv(OUTPUT_DIR / 'convnext_siglip_pseudo_label_audit.csv', index=False)
candidate_table = audit.loc[audit.pseudo_label_candidate, [
    'filepath', 'label_original', 'candidate_target', 'dedup_group', 'fold',
    'convnext_confidence', 'siglip_confidence',
    'p_Lainnya_convnext', 'p_Lainnya_siglip',
    'partition_consensus_rate',
]].copy()
candidate_table.to_csv(OUTPUT_DIR / 'pseudo_label_candidates_review_only.csv', index=False)

summary = {
    'manifest_sha256': GROUPED_MANIFEST_SHA256,
    'siglip_audit_sha256': SIGLIP_AUDIT_SHA256,
    'model_id': MODEL_ID,
    'seed': SEED,
    'folds': FOLDS,
    'fixed_epochs': FIXED_EPOCHS,
    'run_budget_minutes': RUN_BUDGET_MINUTES,
    'final_output_reserve_minutes': FINAL_OUTPUT_RESERVE_MINUTES,
    'estimated_epoch_seconds': ESTIMATED_EPOCH_SECONDS,
    'resource_profile': 'A100 40 GB, 80 GB system RAM, BF16, effective batch 256',
    'physical_batch_size': BATCH_SIZE,
    'gradient_accumulation_steps': GRAD_ACCUM_STEPS,
    'loader_workers': NUM_WORKERS,
    'checkpoint_selection': 'fixed final epoch; validation evaluated once after training',
    'development_rows': int(len(train_frame)),
    'unseen_label_noisy_holdout_rows': holdout_rows,
    'unseen_holdout_group_hash': holdout_group_hash,
    'macro_f1_original_labels': float(f1.mean()),
    'per_class_f1': dict(zip(CLASSES, map(float, f1))),
    'convnext_siglip_agreement': float(audit.cross_model_agreement.mean()),
    'pseudo_label_candidates_review_only': int(audit.pseudo_label_candidate.sum()),
    'holdout_evaluated': False,
    'targets_modified': False,
}
(OUTPUT_DIR / 'teacher_summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
print('Candidate directions:')
print(
    candidate_table.groupby(['label_original', 'candidate_target']).size()
    if len(candidate_table) else 'none'
)
